# Access GloFAS historical river discharge


[![binder](https://mybinder.org/badge.svg)](https://mybinder.org/v2/gh/Simow-az/2026-icar-glofas-training/main?labpath=content/tutorials/access-glofas-historical-data.ipynb)
[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Simow-az/2026-icar-glofas-training/blob/main/content/tutorials/access-glofas-historical-data.ipynb)
[![github](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Simow-az/2026-icar-glofas-training/blob/main/content/tutorials/access-glofas-historical-data.ipynb)

:::{note}
This tutorial keeps the CDS download step optional so that the notebook runs cleanly even without credentials. When you are ready to work with real data, switch `download_from_cds` to `True`.
:::

This tutorial walks through a first GloFAS workflow: define a historical download request, understand the request parameters, and explore river discharge time series with Python.


## Learning objectives 🎯


By the end of this notebook you will be able to:

- identify the key CDS request parameters for the public GloFAS historical dataset;
- prepare a small `cdsapi` request for consolidated river discharge data;
- inspect and plot daily discharge values with `xarray` and `matplotlib`.


## Prepare your environment


The examples below use `numpy`, `xarray`, and `matplotlib`, which are already listed in this repository's `environment.yml`. The optional download step also uses `cdsapi`.

If you want to retrieve data yourself, make sure your CDS credentials are available locally before enabling the download cell.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

try:
    import cdsapi
except ImportError:
    cdsapi = None


## Define a historical data request


In [ ]:
dataset = "cems-glofas-historical"
output_path = Path("data/glofas-historical-example.nc")
request = {
    "system_version": ["version_4_0"],
    "hydrological_model": ["lisflood"],
    "product_type": ["consolidated"],
    "variable": ["river_discharge_in_the_last_24_hours"],
    "hyear": ["2020"],
    "hmonth": ["02"],
    "hday": [f"{day:02d}" for day in range(1, 16)],
    "data_format": "netcdf",
    "download_format": "zip",
}
request


The request above asks for a compact 15-day sample of consolidated river discharge. Using a short period is a good way to confirm that your workflow works before scaling up to larger downloads.


## Optional CDS download


In [ ]:
download_from_cds = False

if download_from_cds:
    if cdsapi is None:
        raise ImportError("Install cdsapi before enabling the download step.")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    client = cdsapi.Client()
    client.retrieve(dataset, request, str(output_path))
    print(f"Saved example file to {output_path}")
else:
    print("Skipping the live download. Enable `download_from_cds` when you have CDS access.")


## Create a lightweight practice dataset


To keep the tutorial runnable everywhere, the next steps create a small synthetic dataset that mimics a few station hydrographs. Replace it with your downloaded file once you are ready to work with real GloFAS output.


In [ ]:
times = np.arange(np.datetime64("2020-02-01"), np.datetime64("2020-02-16"))
stations = ["Upper basin", "Middle basin", "Lower basin"]

base_signal = np.linspace(220, 310, times.size)
seasonal_cycle = 25 * np.sin(np.linspace(0, 2 * np.pi, times.size))
station_offsets = np.array([0, 40, 85])[:, None]

river_discharge = station_offsets + base_signal + seasonal_cycle

practice_ds = xr.Dataset(
    data_vars={
        "river_discharge_in_the_last_24_hours": (("station", "time"), river_discharge),
    },
    coords={
        "station": stations,
        "time": times,
    },
)
practice_ds


## Plot a first hydrograph


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
practice_ds["river_discharge_in_the_last_24_hours"].transpose("time", "station").plot(ax=ax)
ax.set_title("Example daily discharge time series")
ax.set_ylabel("River discharge [m³/s]")
ax.set_xlabel("Date")
ax.grid(True, alpha=0.3)
plt.show()


## Summarise the sample


In [ ]:
summary = xr.Dataset({
    "mean_discharge": practice_ds.river_discharge_in_the_last_24_hours.mean(dim="time"),
    "peak_discharge": practice_ds.river_discharge_in_the_last_24_hours.max(dim="time"),
})
summary.to_dataframe().round(1)


## Take home messages 📌


- Start with a small historical request so you can validate your credentials and request syntax quickly.
- `xarray` gives you a convenient bridge from downloaded NetCDF files to inspection, filtering, and plotting.
- Once the workflow works on a short sample, you can safely expand it to longer periods or larger spatial domains.
